# BGS+: preprocessing

This notebook prepares the BGS+ dataset for the experiments of Milestone 3, in the same way as `dmg777k_preprocessing.ipynb`.

**Before running:** prepare the data as in `data/README.md`. The same steps run without the notebook with `uv run python -m preprocessing.pipeline bgs`.

In [1]:
import pandas as pd

from preprocessing import pipeline
from preprocessing.geometry import geodesic_km

pd.set_option("display.max_colwidth", 70, "display.width", 160)


def short(iri: str) -> str:
    """Last part of an IRI, for display only; the saved tables keep full IRIs."""
    return iri.rstrip("/").rsplit("/", 1)[-1].rsplit("#", 1)[-1]


r = pipeline.run("bgs")
print(f"{len(r.dataset.triples):,} triples loaded, {len(r.triples):,} kept")

916,199 triples loaded, 882,512 kept


## 1. Preparing the data

### 1.1 The pipeline, with fixes

`data/bgs_data_prep.py` has already parsed the data and converted the geometries to WGS84. Here we make the same five changes as for dmg777k.

**Missing values** removed, grouped by text:

In [2]:
removed = r.placeholders_removed.assign(predicate=r.placeholders_removed["predicate"].map(short))
print(f"Triples removed: {removed['triples'].sum():,}")
removed.groupby("object")["triples"].sum().sort_values(ascending=False).to_frame()

Triples removed: 33,687


,triples
object,
Not Entered,33139
Not defined.,396
Unknown.,44
Not known.,39
Not applicable.,23
Not known,7
N/a,7
Unknown,6
Not Applicable,6


BGS+ marks missing values with words, mostly "Not Entered" (33,139 triples). Each would become a hub linking unrelated entities, so we remove them. We also remove the one empty geometry, `POLYGON EMPTY` (2 triples of one map unit, which also has a point). Placeholder *entities*, such as `ShapeType/NotEntered`, stay; they are in the hub list.

**Node identity**: by text (authors) and by text and kind (ours):

In [3]:
r.identity

,By text (authors),By text and kind (ours)
Nodes,273945,333824
Self-loops,58689,0
Nodes where a literal is merged into an entity,58709,0
Nodes where literals of different types are merged,1161,0


Identifying nodes by text alone merges about 60,000 nodes: 58,709 entities with their `identifier` literal (58,689 self-loops), and 1,161 nodes with values of different types, e.g. 0 as integer and as decimal. With text and kind, there are none.

**Geometries**:

In [4]:
print(r.geometry_counts.to_dict())
print("Geometry types:", r.geometries["geometry_type"].value_counts().to_dict())
print(f"Entities with their own geometry: {len(r.locations):,}")

{'WKT literals': 48603, 'Not valid WKT': 0, 'Empty': 0, 'Usable': 48603}
Geometry types: {'Polygon': 23917, 'Point': 21943, 'LineString': 2743}
Entities with their own geometry: 22,784


All 48,603 remaining WKT literals are valid. 22,784 entities have their own geometry. All other nodes, about 262,000 of the 333,824, get a location only through flooding. This includes all 146 labelled rock units, since none of them has its own geometry.

### 1.2 Checks

The same checks as for dmg777k. The distance row tests the code on two Dutch points, so it is the same as there.

In [5]:
r.checks

,authors' pipeline,ours,expected
check,,,
WKT literals recognised as geometries,48603.0,48603.0,"all usable of 48,603"
WKT literals with keywords not in capitals,0.0,0.0,0
Geometry centres outside the country,NaN,0.0,0
"Amsterdam–Utrecht distance, km (true 34.6)",39.4,34.6,34.6
Self-loops,58689.0,0.0,0
Nodes merging an entity and a literal,58709.0,0.0,0
Nodes merging literals of different types,1161.0,0.0,0


The authors' pipeline finds all geometries here, because BGS+ writes them in capitals. It still has 58,689 self-loops and swaps coordinates. Ours has none of these problems.

## 2. Relations

The same table as for dmg777k. It only describes the relations; choosing them is part of the experiments.

In [6]:
loc = r.locations
baseline_km = pd.Series(geodesic_km(loc["lon"], loc["lat"], loc["lon"].mean(), loc["lat"].mean())).median()
print(f"Baseline: median distance of a located entity to the mean position of all of them: {baseline_km:.1f} km")
relations = r.relations.assign(relation=r.relations["relation"].map(short), literal_kind=r.relations["literal_kind"].map(lambda k: k if pd.isna(k) else short(k)))
print(f"{len(relations)} relations, {relations['median_km'].notna().sum()} with a spatial spread")
relations.dropna(subset=["median_km"]).sort_values("median_km")

Baseline: median distance of a located entity to the mean position of all of them: 248.4 km
103 relations, 20 with a spatial spread


,relation,edges,distinct_objects,object,literal_kind,located_subjects,median_km
8,hasBoundingBox,22784,21945,literal,XMLLiteral,1344.0,0.00
9,hasPointLocation,22784,21943,literal,wktLiteral,1348.0,0.00
11,spatial,22783,21958,literal,wktLiteral,1316.0,0.00
10,hasFootprint,22783,21958,literal,wktLiteral,1316.0,0.00
44,hasLocationDescription,5533,5303,literal,@en,358.0,0.08
43,isStratotypeFor,5534,3486,entity,NaN,3312.0,4.05
1,label,58966,58790,literal,@en,43.0,33.91
14,prefLabel,16052,15943,literal,@en,43.0,33.91
18,hasLexicon,14507,154,entity,NaN,14499.0,85.27
15,hasMapCode,14507,110,literal,plain,14506.0,106.01


Baseline: 248 km.

- Spatial: `hasLocationDescription` (0.08 km), `isStratotypeFor` (4 km).
- Weakly spatial: `hasLexicon`, `hasMapCode`, `hasColourRGB` (85–112 km).
- Not spatial: ages, ranks, `type` (117–248 km).

`hasBoundingBox` is partly GML text, not WKT. These entities also have a WKT footprint, so no location is lost.

The full table, saved as `relations.csv`:

In [7]:
relations

,relation,edges,distinct_objects,object,literal_kind,located_subjects,median_km
0,type,83523,6,entity,NaN,22784.0,248.42
1,label,58966,58790,literal,@en,43.0,33.91
2,identifier,58738,58738,literal,@en,NaN,NaN
3,inScheme,34956,25,entity,NaN,17250.0,227.16
4,isDefinedBy,34550,24,entity,NaN,17250.0,227.16
...,...,...,...,...,...,...,...
98,hasSourceOfName,1,1,literal,@en,NaN,NaN
99,hasRoleDescription,1,1,literal,@en,NaN,NaN
100,about,1,1,literal,anyURI,NaN,NaN
101,hasTopConcept,1,1,entity,NaN,NaN,NaN


### Hubs

The nodes with the most edges. In the EDA, flooding through hubs put 50.3% of nodes at one national centroid.

In [8]:
hubs = r.hubs.assign(node=r.hubs["node"].str[-60:], kind=r.hubs["kind"].map(short), main_relation=r.hubs["main_relation"].map(short))
hubs.head(15)

,node,kind,degree,in_degree,out_degree,main_relation,main_relation_share
0,http://www.w3.org/2004/02/skos/core#Concept,entity,83024,83024,0,type,1.000
1,http://data.bgs.ac.uk/ref/Lexicon/NamedRockUnit,entity,23403,23394,9,inScheme,0.500
2,http://data.bgs.ac.uk/ref/625KGeologyMap/Unit,entity,22493,22488,5,isDefinedBy,0.500
3,http://www.w3.org/1999/02/22-rdf-syntax-ns#nil,entity,15235,15235,0,subClassUDCS,0.193
4,http://data.bgs.ac.uk/ref/CurrentStatus/CURRENT,entity,12759,12759,0,hasCurrentStatus,1.000
5,tp://data.bgs.ac.uk/id/EarthMaterialClass/ComponentRank/MAIN,entity,12729,12729,0,hasComponentRank,1.000
6,http://data.bgs.ac.uk/id/Lexicon/ShapeType/NotEntered,entity,11680,11675,5,hasShapeType,1.000
7,http://data.bgs.ac.uk/id/onshoreOffshore/ONSHORE,entity,11101,11101,0,isOnshoreOffshore,1.000
8,"""FORMAL""",plain,10559,10559,0,hasFormality,1.000
9,http://data.bgs.ac.uk/id/Lexicon/Theme/BEDRCK,entity,10026,10020,6,hasTheme,0.999


The largest hub is `skos:Concept` (83,024 edges), followed by scheme nodes, `rdf:nil` and status values. Some are placeholder entities. None of them is a place. Saved as `hubs.csv`.

## 3. Labels and splits

The published splits. All checks should be 0.

In [9]:
display(r.splits)
r.label_checks

split,train,valid,test,total
class,,,,
FLUV,59,15,19,93
GLACI,35,8,10,53
total,94,23,29,146


,count
Nodes labelled more than once,0
Labelled nodes missing from the graph,0
Triples using the target relation (hasLithogenesis),0


146 rock units (93 FLUV, 53 GLACI), with the same class shares in each split. No label leaks into the graph.

## 4. Saving

Outputs in `data/bgs/processed/`:

- `triples.parquet`: subject, predicate, object (node keys), object_kind
- `geometries.parquet`: node, WKT, geometry type, centre (lon, lat)
- `labels.csv`: node, class, split
- `relations.csv`, `hubs.csv`: the tables from Section 2

In [10]:
pipeline.save(r)
for path in sorted(r.dataset.processed_dir.iterdir()):
    print(f"{path.name:20} {path.stat().st_size / 1e6:8.1f} MB")

geometries.parquet       26.7 MB
hubs.csv                  0.0 MB
labels.csv                0.0 MB
relations.csv             0.0 MB
triples.parquet          50.6 MB


The end of yet another noteboook. enjoy your processed data